# 📖 Notebook 3: Replication and Consistency Levels

Cassandra stores multiple copies of your data across nodes (replication) and lets you choose
how many copies must respond before a read or write is considered successful (consistency level).

This is how Cassandra implements the **CAP theorem** trade-off — you get to tune the dial
between consistency and availability on a per-query basis.

## Learning Objectives

By the end of this notebook, you'll understand:
- How replication works (SimpleStrategy vs. NetworkTopologyStrategy)
- What consistency levels mean (ONE, QUORUM, ALL)
- How to tune the consistency-availability trade-off
- Why QUORUM reads + QUORUM writes = strong consistency
- What happens when a node goes down

## 🛠️ Setup

Make sure the 3-node cluster is running:

```bash
cd 03-technologies/databases/cassandra
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
from cassandra.cluster import Cluster
from cassandra.query import SimpleStatement
from cassandra import ConsistencyLevel
from tabulate import tabulate
import time

# max_schema_agreement_wait: after a CREATE TABLE, the schema has to reach
# every node before another node will answer a read against that table.
# The driver default (10s) is not enough on a freshly-formed 3-node ring,
# and the symptom is a confusing
#   ReadFailure ... INCOMPATIBLE_SCHEMA
# on the very first SELECT rather than on the DDL that caused it.
cluster = Cluster(["localhost"], port=9042, max_schema_agreement_wait=60)
session = cluster.connect()

# --- Make DDL deterministic on a multi-node ring -----------------------------
# Cassandra propagates schema changes by gossip. On this 3-node cluster a
# CREATE TABLE followed immediately by a read or write can reach a replica that
# has not seen the new schema yet, and the coordinator answers with a confusing
#   ReadFailure / WriteFailure ... INCOMPATIBLE_SCHEMA
# pointing at a random node. Waiting for every node to agree after each DDL
# statement removes that race, so these notebooks behave the same every run.
# LWT (Paxos) and QUORUM operations on a small local ring can exceed the
# driver default of 10s while the cluster is still warming up.
session.default_timeout = 30
_execute = session.execute


def _execute_and_settle(query, *args, **kwargs):
    """session.execute, hardened against schema gossip still settling.

    Two things happen here:

    1. After DDL we wait for every node to agree on the new schema.
    2. If a query still comes back INCOMPATIBLE_SCHEMA, we retry it. On a ring
       whose third node finished bootstrapping seconds ago, `wait_for_schema_
       agreement` can return while a replica is not yet serving the new table,
       and the coordinator answers Read/WriteFailure naming that node. It is
       transient, and retrying is exactly what a production client does with a
       transient coordinator failure.
    """
    import time as _time

    text = query if isinstance(query, str) else getattr(query, "query_string", "")
    is_ddl = text.strip().upper().startswith(("CREATE", "ALTER", "DROP", "TRUNCATE"))

    last = None
    for attempt in range(8):
        try:
            result = _execute(query, *args, **kwargs)
            if is_ddl:
                cluster.control_connection.wait_for_schema_agreement(wait_time=60)
            return result
        except Exception as exc:
            # INCOMPATIBLE_SCHEMA: a replica has not caught up with the new schema.
            # timed out / CAS operation timed out: a replica was too slow to answer.
            # Both are transient on a ring that has just booted; a real client
            # retries them too. Anything else is a genuine error -- re-raise it.
            transient = ("INCOMPATIBLE_SCHEMA" in str(exc)) or ("timed out" in str(exc).lower())
            if not transient:
                raise
            last = exc
            _time.sleep(2 * (attempt + 1))
    raise RuntimeError(
        f"Schema never settled after 8 retries. Last error: {last}\n"
        "Check `docker compose exec cassandra-node1 nodetool describecluster` -- "
        "all three nodes should report a single schema version."
    ) from last


session.execute = _execute_and_settle


def wait_for_cluster_ready(timeout=180.0):
    """Block until the ring can actually create a table and write to it at ALL.

    `docker compose up --wait` returns as soon as every node reports UN, but a
    node that has only just finished bootstrapping will still reject queries
    for a short while with INCOMPATIBLE_SCHEMA. Rather than sprinkle sleeps
    around, we prove readiness once: create a throwaway RF=3 table, write to it
    at consistency ALL (so every replica must answer), then drop it.
    """
    import time as _time

    from cassandra.query import SimpleStatement as _SimpleStatement
    from cassandra import ConsistencyLevel as _CL

    deadline = _time.time() + timeout
    last = None
    while _time.time() < deadline:
        try:
            _execute(
                "CREATE KEYSPACE IF NOT EXISTS readiness_probe WITH replication = "
                "{'class': 'SimpleStrategy', 'replication_factor': 3}"
            )
            cluster.control_connection.wait_for_schema_agreement(wait_time=60)
            _execute("CREATE TABLE IF NOT EXISTS readiness_probe.ping (id int PRIMARY KEY)")
            cluster.control_connection.wait_for_schema_agreement(wait_time=60)
            _execute(_SimpleStatement(
                "INSERT INTO readiness_probe.ping (id) VALUES (1)",
                consistency_level=_CL.ALL,
            ))
            _execute("DROP KEYSPACE readiness_probe")
            cluster.control_connection.wait_for_schema_agreement(wait_time=60)
            return
        except Exception as exc:  # replica still settling -- back off and retry
            last = exc
            _time.sleep(3)
    raise TimeoutError(
        f"Cassandra ring never became ready within {timeout}s. Last error: {last}\n"
        "Check `docker compose exec cassandra-node1 nodetool status` -- you need 3 UN nodes."
    )


wait_for_cluster_ready()
print("Ring ready: all 3 replicas accept schema changes and ALL-consistency writes")
# -----------------------------------------------------------------------------

print(f"Connected to: {cluster.metadata.cluster_name}")
print(f"\nCluster nodes:")
for host in cluster.metadata.all_hosts():
    print(f"  {host.address} (rack={host.rack}, dc={host.datacenter})")

## Step 1: Replication Strategies

When you create a keyspace, you tell Cassandra **how many copies** of each piece of data to keep.

### SimpleStrategy
Picks replicas by walking clockwise around the token ring. Good for development and single-datacenter.

### NetworkTopologyStrategy
Picks replicas per datacenter, aware of racks. Ensures replicas are on different racks so a rack
failure doesn't lose multiple copies. **Recommended for production.**

Let's create keyspaces with different replication strategies and see the difference.

In [ ]:
# SimpleStrategy: just a replication factor number
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS demo_simple
    WITH REPLICATION = {
        'class': 'SimpleStrategy',
        'replication_factor': 3
    }
""")

# NetworkTopologyStrategy: specify replicas per datacenter
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS demo_network
    WITH REPLICATION = {
        'class': 'NetworkTopologyStrategy',
        'dc1': 3
    }
""")

print("Created two keyspaces:")
print("")
print("demo_simple:  SimpleStrategy, RF=3")
print("  → 3 copies of every row, placed by walking the token ring")
print("")
print("demo_network: NetworkTopologyStrategy, dc1=3")
print("  → 3 copies in dc1, placed on different racks when possible")
print("  → Our nodes are on rack1, rack2, rack3, so each gets one copy")

In [ ]:
# Let's verify replication by checking which nodes own data for a key
session.set_keyspace('demo_simple')

session.execute("""
    CREATE TABLE IF NOT EXISTS replication_test (
        id text PRIMARY KEY,
        value text
    )
""")

session.execute("INSERT INTO replication_test (id, value) VALUES ('key1', 'hello')")

# Use the token metadata to find which nodes own this key
token_map = cluster.metadata.token_map
keyspace_meta = cluster.metadata.keyspaces.get('demo_simple')

print("Replication factor 3 means the data is on these nodes:")
for host in cluster.metadata.all_hosts():
    print(f"  ✅ {host.address} (rack={host.rack})")
print("\nWith RF=3 and 3 nodes, every node has a copy of every row.")
print("This is maximum redundancy — any 2 nodes can fail and you still have the data.")

## Step 2: Consistency Levels Explained

The **consistency level** controls how many replica nodes must respond before a read or write
is considered successful.

| Level | Nodes Required | Speed | Consistency | Availability |
|-------|---------------|-------|-------------|-------------|
| ONE | 1 replica | ⚡ Fastest | Lowest | Highest |
| QUORUM | Majority (N/2 + 1) | Medium | Strong* | Good |
| ALL | All replicas | 🐌 Slowest | Strongest | Lowest |

*QUORUM reads + QUORUM writes = strong consistency (guaranteed to see latest write).

With 3 replicas:
- ONE = 1 node
- QUORUM = 2 nodes (3/2 + 1 = 2)
- ALL = 3 nodes

In [ ]:
session.set_keyspace('demo_simple')

session.execute("""
    CREATE TABLE IF NOT EXISTS consistency_demo (
        id text PRIMARY KEY,
        value text
    )
""")

# Write with different consistency levels and measure latency
results = []

for cl_name, cl in [("ONE", ConsistencyLevel.ONE),
                     ("QUORUM", ConsistencyLevel.QUORUM),
                     ("ALL", ConsistencyLevel.ALL)]:
    stmt = SimpleStatement(
        f"INSERT INTO consistency_demo (id, value) VALUES ('test_{cl_name}', 'data')",
        consistency_level=cl
    )

    # Average over multiple runs for more stable measurements
    times = []
    for _ in range(10):
        start = time.perf_counter()
        session.execute(stmt)
        elapsed = (time.perf_counter() - start) * 1000
        times.append(elapsed)

    avg_ms = sum(times) / len(times)
    results.append([cl_name, f"{avg_ms:.2f} ms", "1 node" if cl_name == "ONE" else "2 nodes" if cl_name == "QUORUM" else "3 nodes"])

print("Write latency by consistency level (average of 10 runs):")
print(tabulate(results, headers=["Level", "Avg Latency", "Nodes Required"], tablefmt="grid"))
print("\nHigher consistency = more nodes must confirm = higher latency")

In [ ]:
# Same exercise for reads
results = []

for cl_name, cl in [("ONE", ConsistencyLevel.ONE),
                     ("QUORUM", ConsistencyLevel.QUORUM),
                     ("ALL", ConsistencyLevel.ALL)]:
    stmt = SimpleStatement(
        "SELECT * FROM consistency_demo WHERE id = 'test_ONE'",
        consistency_level=cl
    )

    times = []
    for _ in range(10):
        start = time.perf_counter()
        session.execute(stmt)
        elapsed = (time.perf_counter() - start) * 1000
        times.append(elapsed)

    avg_ms = sum(times) / len(times)
    results.append([cl_name, f"{avg_ms:.2f} ms"])

print("Read latency by consistency level (average of 10 runs):")
print(tabulate(results, headers=["Level", "Avg Latency"], tablefmt="grid"))

## Step 3: Why QUORUM + QUORUM = Strong Consistency

Here's the key insight that makes Cassandra powerful:

With RF=3 and QUORUM consistency:
- **Write QUORUM** = 2 of 3 nodes confirm the write
- **Read QUORUM** = 2 of 3 nodes respond to the read

Because 2 + 2 > 3, at least **one node** must have participated in both the write and the read.
That node has the latest data, so the read is guaranteed to see the latest write.

```
Write QUORUM (2 of 3):    Node1 ✅  Node2 ✅  Node3 ❌
Read  QUORUM (2 of 3):    Node1 ❌  Node2 ✅  Node3 ✅
                                     ^^^^^
                          Node2 participated in BOTH → has latest data
```

The general formula: **W + R > N** guarantees strong consistency, where:
- W = number of nodes that confirm writes
- R = number of nodes that respond to reads
- N = replication factor

In [ ]:
# Demonstrate: QUORUM write followed by QUORUM read always sees latest value
for i in range(5):
    # Write with QUORUM
    write_stmt = SimpleStatement(
        f"INSERT INTO consistency_demo (id, value) VALUES ('quorum_test', 'version_{i}')",
        consistency_level=ConsistencyLevel.QUORUM
    )
    session.execute(write_stmt)

    # Read with QUORUM immediately after
    read_stmt = SimpleStatement(
        "SELECT value FROM consistency_demo WHERE id = 'quorum_test'",
        consistency_level=ConsistencyLevel.QUORUM
    )
    row = session.execute(read_stmt).one()
    expected = f"version_{i}"
    match = "✅" if row.value == expected else "❌"
    print(f"Write: {expected}  →  Read: {row.value}  {match}")

print("\nQUORUM write + QUORUM read = always see the latest value!")

> 💡 **Need stronger-than-QUORUM guarantees?** For true linearizable uniqueness (e.g., "only one user can claim this username") normal consistency levels aren't enough — you need **Lightweight Transactions (LWT)**, which use the `SERIAL` consistency level and Paxos under the hood. See Notebook 5.

## Step 4: Eventual Consistency with ONE

With consistency level ONE, Cassandra only needs **one node** to confirm. This is fast but
introduces the possibility of **stale reads** — you might read from a node that hasn't
received the latest write yet.

In practice, Cassandra replicates data very quickly (milliseconds), so stale reads are rare.
But if you need guaranteed freshness, use QUORUM.

In [ ]:
# Write with ONE, read with ONE — eventual consistency
# In practice, this usually works fine because replication is fast,
# but there's no GUARANTEE the read sees the latest write.

write_stmt = SimpleStatement(
    "INSERT INTO consistency_demo (id, value) VALUES ('one_test', 'latest_value')",
    consistency_level=ConsistencyLevel.ONE
)
session.execute(write_stmt)

read_stmt = SimpleStatement(
    "SELECT value FROM consistency_demo WHERE id = 'one_test'",
    consistency_level=ConsistencyLevel.ONE
)
row = session.execute(read_stmt).one()
print(f"Write ONE → Read ONE: {row.value}")
print("\nThis worked, but there's no guarantee. With ONE:")
print("  - The write might go to Node1")
print("  - The read might go to Node2 (which hasn't received the write yet)")
print("  - You'd get stale data")
print("\nFor our small local cluster, replication is near-instant so it usually works.")
print("In production with network delays, stale reads are more likely with CL=ONE.")

## Step 5: What Happens When a Node Goes Down?

Let's see how different consistency levels behave when a node is unavailable.

With RF=3:
- **CL=ONE**: Tolerates 2 node failures (only need 1 of 3)
- **CL=QUORUM**: Tolerates 1 node failure (need 2 of 3)
- **CL=ALL**: Tolerates 0 node failures (need all 3)

Let's simulate this by stopping a node.

In [ ]:
import subprocess

print("Before stopping a node — current cluster state:")
for host in cluster.metadata.all_hosts():
    print(f"  {host.address}: {'UP' if host.is_up else 'DOWN'} (rack={host.rack})")

print("\n⚠️ Stopping cassandra-node3...")
subprocess.run(["docker", "stop", "cassandra-node3"], capture_output=True)
time.sleep(5)  # wait for gossip to detect the failure

print("\nNode3 stopped. Let's test each consistency level...")

In [ ]:
# Test reads with different consistency levels while node3 is down
test_cases = [
    ("ONE", ConsistencyLevel.ONE, "Need 1 of 3 — 2 nodes available ✅"),
    ("QUORUM", ConsistencyLevel.QUORUM, "Need 2 of 3 — 2 nodes available ✅"),
    ("ALL", ConsistencyLevel.ALL, "Need 3 of 3 — only 2 available ❌"),
]

for cl_name, cl, explanation in test_cases:
    stmt = SimpleStatement(
        "SELECT * FROM consistency_demo WHERE id = 'test_ONE'",
        consistency_level=cl
    )
    try:
        session.execute(stmt)
        print(f"CL={cl_name:8s} → ✅ Success  ({explanation})")
    except Exception as e:
        error_name = type(e).__name__
        print(f"CL={cl_name:8s} → ❌ Failed   ({explanation})")
        print(f"              Error: {error_name}")

In [ ]:
# Restart node3
print("Restarting cassandra-node3...")
subprocess.run(["docker", "start", "cassandra-node3"], capture_output=True)
print("Node3 restarted. It will rejoin the cluster automatically via gossip.")
print("\nAny writes that happened while it was down will be sent to it via:")
print("  1. Hinted handoff — other nodes saved the writes as 'hints'")
print("  2. Read repair — next time data is read, inconsistencies are fixed")
print("  3. Anti-entropy repair — periodic full-cluster consistency check")

## Step 6: Choosing the Right Consistency Level

Here's a decision guide for system design interviews:

| Use Case | Write CL | Read CL | Why |
|----------|----------|---------|-----|
| Analytics / Logs | ONE | ONE | Speed matters, staleness is OK |
| Social media feed | ONE | ONE | Eventual consistency is fine for feeds |
| Shopping cart | QUORUM | QUORUM | Must see latest cart state |
| Financial ledger | ALL | ALL | Cannot tolerate any inconsistency |
| High-availability API | ONE | ONE | Prefer availability over consistency |

**Rule of thumb**: Start with QUORUM/QUORUM. Weaken to ONE only when you've confirmed
your use case tolerates stale reads.

## 🧠 Key Takeaways

1. **Replication factor** controls how many copies of data exist. RF=3 is the standard.

2. **Consistency level** controls how many nodes must respond per query. It's set per-query, not per-table.

3. **QUORUM write + QUORUM read = strong consistency** because at least one node overlaps.

4. **ONE is fastest** but risks stale reads. **ALL is safest** but can't handle any node failure.

5. **W + R > N** is the formula for strong consistency (W=write nodes, R=read nodes, N=replicas).

6. **NetworkTopologyStrategy** is recommended for production — it spreads replicas across racks/datacenters.

7. When a node goes down, Cassandra uses **hinted handoff** and **read repair** to catch it up when it returns.

## ➡️ Next Notebook

In Notebook 4, we'll explore **compaction strategies** — how Cassandra manages its on-disk storage
using LSM trees, memtables, SSTables, and different compaction strategies.

In [ ]:
cluster.shutdown()
print("Connection closed.")